# Query Rewriter Evaluator

This notebook allows you to test the query rewriter and manually label results as **CORRECT** or **INCORRECT**.

## Features:
- Input previous queries and current query
- Run the intelligent query rewriter
- Manually evaluate if the rewrite is correct
- Store results for analysis
- Export evaluation data

In [ ]:
import sys
import os
import json
from datetime import datetime

sys.path.insert(0, os.path.abspath('..'))

from query_rewriter import rewrite_to_standalone, maybe_rewrite
from services.llm_service import generate_sql

DATABASE_ID = 'degreefyd_online_lms'

print("✓ Query rewriter evaluator loaded")

## Test Case Storage

All test cases and evaluations will be stored here.

In [ ]:
# Store all test cases and evaluations
evaluations = []

def add_evaluation(previous_queries, current_query, rewritten_query, is_correct, notes=""):
    """Add an evaluation to the storage."""
    evaluation = {
        "timestamp": datetime.now().isoformat(),
        "previous_queries": previous_queries,
        "current_query": current_query,
        "rewritten_query": rewritten_query,
        "is_correct": is_correct,
        "notes": notes
    }
    evaluations.append(evaluation)
    return evaluation

def print_evaluations():
    """Print all evaluations."""
    print(f"\n{'='*100}")
    print(f"EVALUATIONS ({len(evaluations)} total)")
    print(f"{'='*100}")
    for i, eval in enumerate(evaluations, 1):
        status = "✓ CORRECT" if eval['is_correct'] else "✗ INCORRECT"
        print(f"\n{i}. {status}")
        print(f"   Previous: {eval['previous_queries']}")
        print(f"   Current:  {eval['current_query']}")
        print(f"   Rewritten: {eval['rewritten_query']}")
        if eval['notes']:
            print(f"   Notes: {eval['notes']}")

def export_evaluations(filename="query_rewriter_evaluations.json"):
    """Export evaluations to JSON file."""
    with open(filename, 'w') as f:
        json.dump(evaluations, f, indent=2)
    print(f"✓ Exported {len(evaluations)} evaluations to {filename}")

print("✓ Evaluation storage initialized")

## Manual Test - Single Query

Use this cell to test a single query pair and manually evaluate the result.

In [ ]:
# INPUT: Configure your test case here
PREVIOUS_QUERIES = [
    "admission today"
]

CURRENT_QUERY = "and forms"

# Run the rewriter
print("="*80)
print("TEST CASE")
print("="*80)
print(f"Previous queries: {PREVIOUS_QUERIES}")
print(f"Current query: {CURRENT_QUERY}")
print()

rewritten = rewrite_to_standalone(PREVIOUS_QUERIES, CURRENT_QUERY)

print(f"\nRewritten query: {rewritten}")
print()

# MANUAL EVALUATION - Change this after reviewing the result
IS_CORRECT = True  # Set to True if correct, False if incorrect
NOTES = ""  # Add any notes about why it's correct/incorrect

# Store the evaluation
eval_result = add_evaluation(PREVIOUS_QUERIES, CURRENT_QUERY, rewritten, IS_CORRECT, NOTES)
status = "✓ CORRECT" if IS_CORRECT else "✗ INCORRECT"
print(f"Evaluation: {status}")
if NOTES:
    print(f"Notes: {NOTES}")

## Batch Test - Multiple Cases

Test multiple cases at once and evaluate each one.

In [ ]:
# INPUT: Configure multiple test cases
TEST_CASES = [
    {
        "previous": ["admission today"],
        "current": "and forms",
        "expected_correct": True,
        "notes": "Should merge to 'admission and forms today'"
    },
    {
        "previous": ["admission and forms today"],
        "current": "and icc done",
        "expected_correct": True,
        "notes": "Should merge to 'admission and forms and icc done today'"
    },
    {
        "previous": ["admission and forms and icc done today"],
        "current": "and ni",
        "expected_correct": True,
        "notes": "Should merge to 'admission and forms and icc done and ni today'"
    },
    {
        "previous": ["what unusual did u experience in yesterday lead flow as compared to its pervious day"],
        "current": "this data in source and campign level",
        "expected_correct": True,
        "notes": "Should add source and campaign grouping"
    },
    {
        "previous": ["show me leads today"],
        "current": "how many admissions yesterday",
        "expected_correct": False,
        "notes": "Should NOT merge - different topic and time"
    },
]

print(f"\nRunning {len(TEST_CASES)} test cases...\n")
print(f"{'#':<4} {'Current Query':<45} {'Rewritten':<55} {'Status'}")
print("-"*120)

for i, test in enumerate(TEST_CASES, 1):
    rewritten = rewrite_to_standalone(test['previous'], test['current'])
    
    # For manual evaluation, review the output and set IS_CORRECT
    print(f"\n{i}. Previous: {test['previous']}")
    print(f"   Current: {test['current']}")
    print(f"   Rewritten: {rewritten}")
    print(f"   Expected: {'CORRECT' if test['expected_correct'] else 'INCORRECT'}")
    print(f"   Notes: {test['notes']}")
    
    # MANUALLY SET THIS after reviewing each result
    is_correct = test['expected_correct']  # Change this based on actual review
    
    add_evaluation(test['previous'], test['current'], rewritten, is_correct, test['notes'])
    
    status = "✓" if is_correct else "✗"
    print(f"   {status} Evaluated as: {'CORRECT' if is_correct else 'INCORRECT'}")

print(f"\n✓ Completed {len(TEST_CASES)} test cases")

## View All Evaluations

In [ ]:
print_evaluations()

## Calculate Accuracy

In [ ]:
if evaluations:
    total = len(evaluations)
    correct = sum(1 for e in evaluations if e['is_correct'])
    accuracy = (correct / total) * 100
    
    print(f"\n{'='*60}")
    print(f"ACCURACY REPORT")
    print(f"{'='*60}")
    print(f"Total evaluations: {total}")
    print(f"Correct: {correct}")
    print(f"Incorrect: {total - correct}")
    print(f"Accuracy: {accuracy:.1f}%")
    print(f"{'='*60}")
else:
    print("No evaluations yet. Run some test cases first.")

## Export Evaluations

In [ ]:
# Export to JSON file
export_evaluations()

## Clear Evaluations

Use this to start fresh (careful - this deletes all stored evaluations)

In [ ]:
# Uncomment to clear all evaluations
# evaluations.clear()
# print("✓ Cleared all evaluations")

## Interactive Test Loop

Run this cell to enter an interactive loop where you can input queries one by one.

In [ ]:
def interactive_test():
    """Interactive testing loop."""
    session_queries = []
    
    print("\n" + "="*80)
    print("INTERACTIVE QUERY REWRITER TESTER")
    print("="*80)
    print("Commands:")
    print("  - Type a query to test it")
    print("  - 'clear' - Clear session history")
    print("  - 'history' - Show session history")
    print("  - 'eval' - Show all evaluations")
    print("  - 'export' - Export evaluations")
    print("  - 'quit' - Exit")
    print("-"*80)
    
    while True:
        query = input("\nEnter query (or command): ").strip()
        
        if not query:
            continue
        
        if query.lower() == 'quit':
            print("Exiting...")
            break
        
        if query.lower() == 'clear':
            session_queries.clear()
            print("✓ Session history cleared")
            continue
        
        if query.lower() == 'history':
            print(f"\nSession history ({len(session_queries)} queries):")
            for i, q in enumerate(session_queries, 1):
                print(f"  {i}. {q}")
            continue
        
        if query.lower() == 'eval':
            print_evaluations()
            continue
        
        if query.lower() == 'export':
            export_evaluations()
            continue
        
        # Run the rewriter
        rewritten = rewrite_to_standalone(session_queries, query)
        
        print(f"\n{'='*80})
        print(f"Previous queries: {session_queries}")
        print(f"Current query: {query}")
        print(f"Rewritten query: {rewritten}")
        print(f"{'='*80})
        
        # Manual evaluation
        eval_input = input("\nIs this correct? (y/n): ").strip().lower()
        is_correct = eval_input == 'y'
        
        notes = input("Notes (optional, press Enter to skip): ").strip()
        
        add_evaluation(session_queries.copy(), query, rewritten, is_correct, notes)
        
        status = "✓ CORRECT" if is_correct else "✗ INCORRECT"
        print(f"\nEvaluation: {status}")
        
        # Add to session history
        session_queries.append(rewritten if is_correct else query)

# Uncomment to run interactive tester
# interactive_test()
print("\nTo run interactive tester, uncomment the last line in this cell.")